# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadShayan8401/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


One row represents the search performance of a single content item (`content_hash_id`) for one client (`client_hash_id`) over the selected analysis period.

For this project, I use a 90-day observation window divided into two 45-day periods. The first 45 days are used to generate historical features, while the second 45 days are used to determine whether content performance declined.

This design ensures that only information available before the prediction point is used for feature engineering, reducing the risk of data leakage.

In [11]:
import duckdb
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

# Connect to DuckDB
con = duckdb.connect()

# Create Hugging Face secret
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Define dataset location
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Verify the selected analysis month
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
"""

result = con.sql(query).df()
display(result)

HF Token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,clients,content_items,start_date,end_date
0,9841378,55,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

- imp_prev45
- clk_prev45
- pos_prev45
- visible_queries
- position_volatility

These are historical features available before making a refresh decision.

## Label

is_declining

The label is 1 if impressions decrease by at least 20% in the later 45-day period.

## Context

- client_hash_id
- content_hash_id
- report_date

These identify the client, page and reporting period.

## Excluded

- imp_last45
- clk_last45

These fields come from the future prediction window and would cause target leakage if used as features.

In [4]:
features = [
    "imp_prev45",
    "clk_prev45",
    "pos_prev45",
    "visible_queries",
    "position_volatility"
]

print("Features")

for f in features:
    print(f)

Features
imp_prev45
clk_prev45
pos_prev45
visible_queries
position_volatility


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification Queries

The following queries verify the assumptions made in the data contract.

1. Verify the grain.
2. Verify the selected month.
3. Verify data availability.

In [5]:
# Query 1

grain = con.sql(f"""
SELECT
client_hash_id,
content_hash_id,
report_date
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN DATE '2026-03-01'
AND DATE '2026-03-31'
LIMIT 10
""").df()

print("Query 1")
display(grain)

# Query 2

counts = con.sql(f"""
SELECT
COUNT(*) AS rows,
COUNT(DISTINCT client_hash_id) AS clients,
COUNT(DISTINCT content_hash_id) AS pages,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN DATE '2026-03-01'
AND DATE '2026-03-31'
""").df()

print("Query 2")
display(counts)

# Query 3

availability = con.sql(f"""
SELECT
COUNT(*) AS clients_with_gsc
FROM {TABLES['dim_clients']}
WHERE has_gsc_access IS TRUE
""").df()

print("Query 3")
display(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1


,client_hash_id,content_hash_id,report_date
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 2


,rows,clients,pages,start_date,end_date
0,9841378,55,331437,2026-03-01,2026-03-31


Query 3


,clients_with_gsc
0,67


## Feature Frame

The following five features are engineered from historical March 2026 data.

- Impressions
- Clicks
- Average Position
- Position Volatility
- Visible Query Count

In [12]:
feature_frame = con.sql(f"""
WITH march AS (

SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    STDDEV(gsc_avg_position) AS position_volatility

FROM {TABLES['fact_daily']}

WHERE report_date BETWEEN DATE '2026-03-01'
AND DATE '2026-03-31'

GROUP BY
    client_hash_id,
    content_hash_id

),

queries AS (

SELECT
    content_hash_id,
    ANY_VALUE(content_visible_query_count) AS visible_queries

FROM {TABLES['fact_query_90d']}

GROUP BY content_hash_id

)

SELECT
    m.*,
    q.visible_queries

FROM march m

LEFT JOIN queries q
ON m.content_hash_id = q.content_hash_id

LIMIT 10
""").df()

display(feature_frame)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,avg_position,position_volatility,visible_queries
0,client_08a6a72ff48e62c0,content_5f6fae04728d32ab,251.0,1.0,30.656492,24.599786,5
1,client_08a6a72ff48e62c0,content_5f71205e0b46f70a,1879.0,15.0,4.020550,1.665316,12
2,client_08a6a72ff48e62c0,content_5f716493f7989b45,1721.0,2.0,5.037140,1.882591,17
3,client_08a6a72ff48e62c0,content_5f8b67a6b0494e15,104.0,0.0,37.556047,19.827055,1
4,client_08a6a72ff48e62c0,content_5f93f846d9834caf,754.0,3.0,11.288876,7.991780,2
5,client_08a6a72ff48e62c0,content_5f9eb08a6275de55,116.0,0.0,72.253513,18.243947,1
6,client_08a6a72ff48e62c0,content_5fb5bd16c7ad75f1,198.0,0.0,46.649377,14.877794,8
7,client_08a6a72ff48e62c0,content_5fb962ebc3de35af,7.0,0.0,7.285714,5.794086,4
8,client_08a6a72ff48e62c0,content_5fba838beb063866,136.0,0.0,71.876222,15.970159,7
9,client_08a6a72ff48e62c0,content_5fbc3a12b51e755f,59.0,0.0,80.421014,13.422689,5


### Feature Availability

**Impressions** – available because they are collected before the prediction date.

**Clicks** – historical Search Console data.

**Average Position** – calculated from historical rankings.

**Position Volatility** – computed only from previous rankings.

**Visible Query Count** – known from historical query data before prediction.

## Leakage Demonstration

A future-derived feature is intentionally added to demonstrate how target leakage can artificially improve model performance. The feature is then removed.

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# Create a simple label from the data
feature_frame["is_popular"] = (
    feature_frame["impressions"] >
    feature_frame["impressions"].median()
).astype(int)

# -------------------------------
# Honest model (without leakage)
# -------------------------------

X = feature_frame[
    ["clicks", "avg_position", "position_volatility", "visible_queries"]
].fillna(0)

y = feature_frame["is_popular"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

honest_score = model.score(X_test, y_test)
print("Honest model accuracy:", round(honest_score, 3))

# -------------------------------
# Deliberate leakage
# -------------------------------

X_leak = X.copy()
X_leak["leak_feature"] = y

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak,
    y,
    test_size=0.25,
    random_state=42
)

model.fit(X_train_l, y_train_l)

leak_score = model.score(X_test_l, y_test_l)
print("Accuracy with leakage:", round(leak_score, 3))

# -------------------------------
# Remove leakage feature
# -------------------------------

X_leak = X_leak.drop(columns=["leak_feature"])

print("Leakage feature removed.")

Honest model accuracy: 0.667
Accuracy with leakage: 1.0
Leakage feature removed.


### Leakage Lesson

I deliberately added a feature derived from the target (`leak_feature`). This caused the model's accuracy to become unrealistically high because the model had access to information that would not be available at prediction time. After removing the leaked feature, the model's performance returned to a more realistic value. This demonstrates why target leakage must always be checked before trusting model results.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- Clients have different lengths of historical data.
- Some clients have only Search Console data.
- External events are not captured.
- Search performance is observational rather than causal.
- Results should be used for decision support, not proof of causation.

In [13]:
limits = con.sql(f"""
SELECT
COUNT(*) AS total_clients,
SUM(CASE WHEN has_gsc_access THEN 1 ELSE 0 END) AS gsc_clients,
SUM(CASE WHEN has_ga4_access THEN 1 ELSE 0 END) AS ga4_clients
FROM {TABLES['dim_clients']}
""").df()

display(limits)

,total_clients,gsc_clients,ga4_clients
0,104,67.0,54.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.